# 02 - Fetch Weather Data

Fetches hourly weather observations for the DWD (Deutscher Wetterdienst)
station nearest to Münster — **Münster/Osnabrück (airport), station id
`01766`** — from the [DWD Open Data portal](https://opendata.dwd.de/), saves
them under `data/raw/weather/`, and reports:

- date coverage per parameter (min/max timestamp, number of records)
- the data's actual resolution
- an **explicit count and listing of missing hourly timestamps** (gaps are
  never silently dropped)

Three parameters are fetched, all plausible candidates for a bike-traffic
model: **air temperature + relative humidity**, **precipitation**, and
**wind speed + direction**.

All fetching and schema validation happens in
`src/muenster_bike_forecast/data/weather.py`; this notebook only
orchestrates calls and reports results. Re-running this notebook is safe:
each parameter's output file is rebuilt deterministically (sorted,
deduplicated, one fixed filename per parameter/station), so it never
accumulates duplicate or stale files.

**Open question / known limitation — resolution mismatch:** DWD hourly
data is genuinely hourly, while bike counts are 15-minute. This notebook
deliberately does **not** resample or align the two — that is left to a
later feature-engineering step, once the modeling approach is decided.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.data.weather import (
    DEFAULT_STATION_ID,
    DEFAULT_STATION_NAME,
    PARAMETER_SPECS,
    WeatherFetchError,
    WeatherSchemaError,
    fetch_weather_history,
    find_missing_hours,
    save_raw_weather,
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "weather"
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

print(f"Station: {DEFAULT_STATION_ID} ({DEFAULT_STATION_NAME})")
print(f"Parameters: {list(PARAMETER_SPECS)}")

Station: 01766 (Münster/Osnabrück)
Parameters: ['air_temperature', 'precipitation', 'wind']


## 1. Fetch and save raw observations for every parameter

For each parameter, fetches the full available DWD record for the station
by combining the frozen `historical` product with the rolling `recent`
product (see `fetch_weather_history`), then saves it as one CSV under
`data/raw/weather/dwd_<parameter>_<station_id>.csv`.

Any parameter whose data fails schema validation raises `WeatherSchemaError`
and stops the notebook here (fail loudly rather than silently using bad
data) — per the project's data-engineering conventions, malformed source
content must never be silently dropped.

In [2]:
weather_frames: dict[str, pd.DataFrame] = {}

for parameter in PARAMETER_SPECS:
    print(f"Fetching {parameter} for station {DEFAULT_STATION_ID}...")
    try:
        df = fetch_weather_history(parameter, station_id=DEFAULT_STATION_ID)
    except (WeatherFetchError, WeatherSchemaError) as exc:
        print(f"  FAILED: {exc}")
        raise
    path = save_raw_weather(df, parameter, DEFAULT_STATION_ID, RAW_DATA_DIR)
    print(f"  {len(df):,} records -> {path.relative_to(PROJECT_ROOT)}")
    weather_frames[parameter] = df

Fetching air_temperature for station 01766...


  322,648 records -> data\raw\weather\dwd_air_temperature_01766.csv
Fetching precipitation for station 01766...


  269,724 records -> data\raw\weather\dwd_precipitation_01766.csv
Fetching wind for station 01766...


  388,941 records -> data\raw\weather\dwd_wind_01766.csv


## 2. First look: date coverage and resolution

For each parameter: first/last timestamp, number of records, and the
median gap between consecutive timestamps (the empirical resolution).

In [3]:
coverage_rows = []
for parameter, df in weather_frames.items():
    timestamps = df["timestamp"].sort_values()
    median_gap = timestamps.diff().median()
    coverage_rows.append(
        {
            "parameter": parameter,
            "n_records": len(df),
            "first_timestamp": timestamps.min(),
            "last_timestamp": timestamps.max(),
            "median_resolution": median_gap,
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
coverage_df

,parameter,n_records,first_timestamp,last_timestamp,median_resolution
0,air_temperature,322648,1989-10-01 07:00:00+00:00,2026-07-22 23:00:00+00:00,0 days 01:00:00
1,precipitation,269724,1995-09-01 00:00:00+00:00,2026-07-22 23:00:00+00:00,0 days 01:00:00
2,wind,388941,1982-01-01 00:00:00+00:00,2026-07-22 23:00:00+00:00,0 days 01:00:00


## 3. Missing timestamp report

DWD's own hourly grid can have gaps (station outages, transmission
issues). `find_missing_hours` computes every missing hour within each
parameter's own min/max coverage window — gaps are reported explicitly,
never silently dropped.

In [4]:
gap_summary_rows = []
gap_frames: dict[str, pd.DataFrame] = {}

for parameter, df in weather_frames.items():
    missing = find_missing_hours(df)
    gap_frames[parameter] = missing
    span_hours = (df["timestamp"].max() - df["timestamp"].min()) / pd.Timedelta(hours=1)
    gap_summary_rows.append(
        {
            "parameter": parameter,
            "n_missing_hours": len(missing),
            "pct_missing": 100 * len(missing) / span_hours if span_hours else float("nan"),
        }
    )

gap_summary_df = pd.DataFrame(gap_summary_rows)
gap_summary_df

,parameter,n_missing_hours,pct_missing
0,air_temperature,1,0.000310
1,precipitation,1068,0.394400
2,wind,1635,0.418614


In [5]:
# Show the first few missing timestamps per parameter, for a concrete look
# at where the gaps actually are (not just a count).
for parameter, missing in gap_frames.items():
    print(f"{parameter}: {len(missing)} missing hours")
    if not missing.empty:
        print(missing.head(5).to_string(index=False))
    print()

air_temperature: 1 missing hours
                timestamp
2005-05-29 10:00:00+00:00

precipitation: 1068 missing hours
                timestamp
1995-09-02 10:00:00+00:00
1995-09-11 22:00:00+00:00
1995-09-11 23:00:00+00:00
1995-09-18 15:00:00+00:00
1995-09-21 14:00:00+00:00

wind: 1635 missing hours
                timestamp
1982-07-01 00:00:00+00:00
1982-07-01 01:00:00+00:00
1982-07-01 02:00:00+00:00
1982-07-01 03:00:00+00:00
1982-07-01 04:00:00+00:00



## 4. Value-level missing data (sentinel `-999` → NA)

Separately from *timestamp* gaps (an hour with no row at all), DWD also
marks individual missing *values* within an otherwise-present hourly row
using the sentinel `-999`, which `fetch_weather_history` already converts
to `NA`. This reports how many such value-level gaps exist per column.

In [6]:
for parameter, df in weather_frames.items():
    value_columns = [c for c in df.columns if c not in ("station_id", "timestamp")]
    na_counts = df[value_columns].isna().sum()
    print(f"{parameter}:")
    print(na_counts.to_string())
    print()

air_temperature:
quality_level             0
air_temperature_c        40
relative_humidity_pct    97

precipitation:
quality_level                   0
precipitation_mm              178
precipitation_indicator       178
precipitation_form         110579

wind:
quality_level           0
wind_speed_ms         159
wind_direction_deg    445



## Open question for later feature engineering

Bike counts are recorded at **15-minute** resolution; this DWD weather
data is at **hourly** resolution (confirmed above via the median gap
between consecutive timestamps). Combining the two datasets will require
deciding how to align them — e.g. forward-filling each hourly weather
value across its four 15-minute sub-intervals, or aggregating bike counts
up to hourly — which is intentionally **out of scope here** and left for a
feature-engineering notebook once the modeling approach is chosen.